In [ ]:
from pathlib import Path

import duckdb as db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Detect project root robustly.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name == "archive":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if PROJECT_ROOT.name != "Multidisciplinary_CBL":
    raise RuntimeError(
        "Could not detect project root. Please run this notebook from the project root or the notebooks/ directory."
    )

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

PARQUET_PATH_ALL = PROCESSED_DIR / "crimes_clean_dedup_all_years.parquet"
PARQUET_SQL_PATH_ALL = PARQUET_PATH_ALL.as_posix()

PARQUET_PATH_FILTERED = PROCESSED_DIR / "crimes_filtered_for_model.parquet"
PARQUET_SQL_PATH_FILTERED = PARQUET_PATH_FILTERED.as_posix()

# true to output the figures
EXPORT_OUTPUTS = False

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Crime parquet path: {PARQUET_PATH_ALL}")
print(f"Parquet exists: {PARQUET_PATH_ALL.exists()}")
print(f"Filtered parquet path: {PARQUET_PATH_FILTERED}")
print(f"Filtered parquet exists: {PARQUET_PATH_FILTERED.exists()}")

if not PARQUET_PATH_ALL.exists():
    raise FileNotFoundError(
        "Could not find crimes_clean_dedup_all_years.parquet. "
        "Expected it under data/processed/."
    )

if not PARQUET_PATH_FILTERED.exists():
    raise FileNotFoundError(
        "Could not find crimes_filtered_for_model.parquet. "
        "Expected it under data/processed/."
    )




Project root: /home/rovsi/Courses/Multidisciplinary_CBL
Crime parquet path: /home/rovsi/Courses/Multidisciplinary_CBL/data/processed/crimes_clean_dedup_all_years.parquet
Parquet exists: True
Filtered parquet path: /home/rovsi/Courses/Multidisciplinary_CBL/data/processed/crimes_filtered_for_model.parquet
Filtered parquet exists: True


In [16]:
CCHI_PATH = PROJECT_ROOT / "data" / "raw" / "Cambridge-CCHI-2026-update.xlsx"
CCHI_SQL_PATH = CCHI_PATH.as_posix()

In [21]:
CCHI_SQL_PATH = CCHI_PATH.as_posix()


df_CCHI_unique_sub_crimes = db.sql(f"""
    WITH cchi AS (
        SELECT
            "Group",
            TRY_CAST("CCHI Score" AS DOUBLE) AS cchi_score
        FROM read_xlsx(
            '{CCHI_SQL_PATH}',
            sheet = 'CCHI 2026 values sheet',
            header = true,
            all_varchar = true
        )
    )

    SELECT
        "Group",
        AVG(cchi_score) AS avg_cchi_score
    FROM cchi
    WHERE cchi_score IS NOT NULL
    GROUP BY "Group"
    ORDER BY avg_cchi_score DESC
""").df()
df_CCHI_unique_sub_crimes

,GROUP,avg_cchi_score
0,VIOLENCE AGAINST THE PERSON,689.898490
1,SEXUAL OFFENCES,651.248387
2,POSSESSION OF WEAPONS,541.150000
3,NaN,365.001712
4,ROBBERY,365.000000
5,BURGLARY,280.705882
6,DRUG OFFENCES,156.448598
7,ARSON AND CRIMINAL DAMAGE,97.911765
8,MISCELLANEOUS CRIMES AGAINST SOCIETY,73.717241
9,NFIB FRAUD,67.833333
